In [12]:
!pip install transformers datasets peft accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.4 MB/s eta 0:00:00


# Task 1

In [2]:
# Core dependencies
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch import nn
import json, collections

# For pretty tree display
try:
    from rich.tree import Tree
    from rich import print as rprint
    RICH = True
except ImportError:
    RICH = False
    print("rich not found — falling back to plain text tree")

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

PyTorch 2.10.0+cu128 | CUDA: True


In [3]:
from abc import ABC, abstractmethod
import torch.nn as nn

In [4]:
class BaseModelHandler(ABC):
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.model = None

    @abstractmethod
    def load_model(self):
        pass

    @abstractmethod
    def get_model(self) -> nn.Module:
        pass

    @abstractmethod
    def generate(self, prompt: str) -> str:
        pass

class BaseParser(ABC):

    @abstractmethod
    def parse(self, model: nn.Module, model_name: str) -> dict:
        pass

    @abstractmethod
    def render(self, parsed: dict):
        pass

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

class HuggingFaceModelHandler(BaseModelHandler):

    def __init__(self, model_name: str):
        super().__init__(model_name)
        self.tokenizer = None

    def load_model(self):
        try:
            from transformers import BitsAndBytesConfig

            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16
            )

            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                quantization_config=bnb_cfg,
                device_map="auto",
            )

        except Exception as e:
            print("Falling back to CPU:", e)

            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                dtype=torch.float16
            )

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model.eval()

    def get_model(self):
        return self.model

    def generate(self, prompt: str):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(**inputs, max_new_tokens=50)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

In [6]:
from rich.tree import Tree
from rich import print as rprint
import collections
RICH = True


class ModelParser(BaseParser):

    def _param_count(self, module: nn.Module) -> int:
        return sum(p.numel() for p in module.parameters())

    def _tensor_shape(self, module: nn.Module) -> dict:
        shapes = {}
        for name, param in module.named_parameters(recurse=False):
            shapes[name] = list(param.shape)
        return shapes

    def parse_module(self, module, name="root", depth=0, max_depth=6):
        node = {
            "name": name,
            "type": module.__class__.__name__,
            "params": self._param_count(module),
        }

        children = list(module.named_children())

        if not children or depth >= max_depth:
            shapes = self._tensor_shape(module)
            if shapes:
                node["shapes"] = shapes
        else:
            node["children"] = [
                self.parse_module(child, child_name, depth+1, max_depth)
                for child_name, child in children
            ]

        return node

    def parse(self, model: nn.Module, model_name: str) -> dict:
        total = self._param_count(model)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

        dtypes = set(p.dtype for p in model.parameters())
        devices = set(str(p.device) for p in model.parameters())

        return {
            "model_name": model_name,
            "total_params": total,
            "trainable_params": trainable,
            "dtypes": [str(d) for d in dtypes],
            "devices": list(devices),
            "architecture": self.parse_module(model),
        }

    def _fmt_params(self, n: int) -> str:
        if n >= 1_000_000: return f"{n/1e6:.2f}M"
        if n >= 1_000: return f"{n/1e3:.1f}K"
        return str(n)

    def _build_rich_tree(self, node, tree):
        for child in node.get("children", []):
            shapes = child.get("shapes", {})
            shape_str = ""

            if shapes:
                shape_str = " → " + ", ".join(
                    f"{k}:{v}" for k, v in shapes.items()
                )

            label = (
                f"[bold cyan]{child['name']}[/] "
                f"[dim]({child['type']})[/] "
                f"[yellow]{self._fmt_params(child['params'])}[/]"
                + f"[dim]{shape_str}[/]"
            )

            branch = tree.add(label)
            self._build_rich_tree(child, branch)

    def render(self, parsed: dict):
        arch = parsed["architecture"]

        header = (
            f"{parsed['model_name']} | "
            f"total: {self._fmt_params(parsed['total_params'])} | "
            f"trainable: {self._fmt_params(parsed['trainable_params'])}"
        )

        tree = Tree(f"[bold]{header}[/]")
        self._build_rich_tree(arch, tree)
        rprint(tree)

    def _collect_layer_types(self, node: dict, acc=None):
        if acc is None:
            acc = collections.Counter()

        acc[node["type"]] += 1

        for c in node.get("children", []):
            self._collect_layer_types(c, acc)

        return acc

    def compare_models(self, *parsed_list):
        print(f"{'Metric':<30}" + "  ".join(
            f"{p['model_name']:>25}" for p in parsed_list
        ))
        print("-" * 80)

        rows = [
            ("Total params",    lambda p: self._fmt_params(p["total_params"])),
            ("Trainable params", lambda p: self._fmt_params(p["trainable_params"])),
            ("Dtype(s)",         lambda p: ", ".join(p["dtypes"])),
            ("Device(s)",        lambda p: ", ".join(p["devices"])),
        ]

        for label, fn in rows:
            print(f"{label:<30}" + "  ".join(
                f"{fn(p):>25}" for p in parsed_list
            ))

        print("\nLayer-type counts:")

        counters = [self._collect_layer_types(p["architecture"]) for p in parsed_list]
        all_types = sorted(set().union(*counters))

        for t in all_types:
            print(f"  {t:<28}" + "  ".join(
                f"{c[t]:>25}" for c in counters
            ))

In [17]:
# Parser
parser = ModelParser()

model_handler_tinyllama = HuggingFaceModelHandler("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model_handler_tinyllama.load_model()
model_tinyllama = model_handler_tinyllama.get_model()
parsed_tinyllama = parser.parse(model_tinyllama, "TinyLlama-1.1B-Chat-v1.0")
parser.render(parsed_tinyllama)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

TinyLlama-1.1B-Chat-v1.0 | total: 615.61M | trainable: 131.16M
├── model (LlamaModel) 550.07M
│   ├── embed_tokens (Embedding) 65.54M → weight:[32000, 2048]
│   ├── layers (ModuleList) 484.53M
│   │   ├── 0 (LlamaDecoderLayer) 22.02M
│   │   │   ├── self_attn (LlamaAttention) 4.72M
│   │   │   │   ├── q_proj (Linear4bit) 2.10M → weight:[2097152, 1]
│   │   │   │   ├── k_proj (Linear4bit) 262.1K → weight:[262144, 1]
│   │   │   │   ├── v_proj (Linear4bit) 262.1K → weight:[262144, 1]
│   │   │   │   └── o_proj (Linear4bit) 2.10M → weight:[2097152, 1]
│   │   │   ├── mlp (LlamaMLP) 17.30M
│   │   │   │   ├── gate_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   ├── up_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   ├── down_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   └── act_fn (SiLUActivation) 0
│   │   │   ├── input_layernorm (LlamaRMSNorm) 2.0K → weight:[2048]
│   │   │   └── post_attention_layernorm (LlamaRMSNorm) 2.0K → weight:[2048]
│   │   ├── 1 (LlamaDecoderLayer) 22.02M
│   │   │   ├── self_attn (LlamaAttention) 4.72M
│   │   │   │   ├── q_proj (Linear4bit) 2.10M → weight:[2097152, 1]
│   │   │   │   ├── k_proj (Linear4bit) 262.1K → weight:[262144, 1]
│   │   │   │   ├── v_proj (Linear4bit) 262.1K → weight:[262144, 1]
│   │   │   │   └── o_proj (Linear4bit) 2.10M → weight:[2097152, 1]
│   │   │   ├── mlp (LlamaMLP) 17.30M
│   │   │   │   ├── gate_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   ├── up_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   ├── down_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   └── act_fn (SiLUActivation) 0
│   │   │   ├── input_layernorm (LlamaRMSNorm) 2.0K → weight:[2048]
│   │   │   └── post_attention_layernorm (LlamaRMSNorm) 2.0K → weight:[2048]
│   │   ├── 2 (LlamaDecoderLayer) 22.02M
│   │   │   ├── self_attn (LlamaAttention) 4.72M
│   │   │   │   ├── q_proj (Linear4bit) 2.10M → weight:[2097152, 1]
│   │   │   │   ├── k_proj (Linear4bit) 262.1K → weight:[262144, 1]
│   │   │   │   ├── v_proj (Linear4bit) 262.1K → weight:[262144, 1]
│   │   │   │   └── o_proj (Linear4bit) 2.10M → weight:[2097152, 1]
│   │   │   ├── mlp (LlamaMLP) 17.30M
│   │   │   │   ├── gate_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   ├── up_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   ├── down_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   └── act_fn (SiLUActivation) 0
│   │   │   ├── input_layernorm (LlamaRMSNorm) 2.0K → weight:[2048]
│   │   │   └── post_attention_layernorm (LlamaRMSNorm) 2.0K → weight:[2048]
│   │   ├── 3 (LlamaDecoderLayer) 22.02M
│   │   │   ├── self_attn (LlamaAttention) 4.72M
│   │   │   │   ├── q_proj (Linear4bit) 2.10M → weight:[2097152, 1]
│   │   │   │   ├── k_proj (Linear4bit) 262.1K → weight:[262144, 1]
│   │   │   │   ├── v_proj (Linear4bit) 262.1K → weight:[262144, 1]
│   │   │   │   └── o_proj (Linear4bit) 2.10M → weight:[2097152, 1]
│   │   │   ├── mlp (LlamaMLP) 17.30M
│   │   │   │   ├── gate_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   ├── up_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   ├── down_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   └── act_fn (SiLUActivation) 0
│   │   │   ├── input_layernorm (LlamaRMSNorm) 2.0K → weight:[2048]
│   │   │   └── post_attention_layernorm (LlamaRMSNorm) 2.0K → weight:[2048]
│   │   ├── 4 (LlamaDecoderLayer) 22.02M
│   │   │   ├── self_attn (LlamaAttention) 4.72M
│   │   │   │   ├── q_proj (Linear4bit) 2.10M → weight:[2097152, 1]
│   │   │   │   ├── k_proj (Linear4bit) 262.1K → weight:[262144, 1]
│   │   │   │   ├── v_proj (Linear4bit) 262.1K → weight:[262144, 1]
│   │   │   │   └── o_proj (Linear4bit) 2.10M → weight:[2097152, 1]
│   │   │   ├── mlp (LlamaMLP) 17.30M
│   │   │   │   ├── gate_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   ├── up_proj (Linear4bit) 5.77M → weight:[5767168, 1]
│   │   │   │   ├── down_proj (Linear4bit) 5.77M → weight:[

In [18]:
model_handler_phi = HuggingFaceModelHandler("microsoft/Phi-4-mini-instruct")
model_handler_phi.load_model()
model_phi = model_handler_phi.get_model()
parsed_phi = parser.parse(model_phi, "Phi-4-mini-instruct")
parser.render(parsed_phi)

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

Phi-4-mini-instruct | total: 2225.41M | trainable: 614.80M
├── model (Phi3Model) 2225.41M
│   ├── embed_tokens (Embedding) 614.60M → weight:[200064, 3072]
│   ├── layers (ModuleList) 1610.81M
│   │   ├── 0 (Phi3DecoderLayer) 50.34M
│   │   │   ├── self_attn (Phi3Attention) 12.58M
│   │   │   │   ├── o_proj (Linear4bit) 4.72M → weight:[4718592, 1]
│   │   │   │   └── qkv_proj (Linear4bit) 7.86M → weight:[7864320, 1]
│   │   │   ├── mlp (Phi3MLP) 37.75M
│   │   │   │   ├── gate_up_proj (Linear4bit) 25.17M → weight:[25165824, 1]
│   │   │   │   ├── down_proj (Linear4bit) 12.58M → weight:[12582912, 1]
│   │   │   │   └── activation_fn (SiLUActivation) 0
│   │   │   ├── input_layernorm (Phi3RMSNorm) 3.1K → weight:[3072]
│   │   │   ├── post_attention_layernorm (Phi3RMSNorm) 3.1K → weight:[3072]
│   │   │   ├── resid_attn_dropout (Dropout) 0
│   │   │   └── resid_mlp_dropout (Dropout) 0
│   │   ├── 1 (Phi3DecoderLayer) 50.34M
│   │   │   ├── self_attn (Phi3Attention) 12.58M
│   │   │   │   ├── o_proj (Linear4bit) 4.72M → weight:[4718592, 1]
│   │   │   │   └── qkv_proj (Linear4bit) 7.86M → weight:[7864320, 1]
│   │   │   ├── mlp (Phi3MLP) 37.75M
│   │   │   │   ├── gate_up_proj (Linear4bit) 25.17M → weight:[25165824, 1]
│   │   │   │   ├── down_proj (Linear4bit) 12.58M → weight:[12582912, 1]
│   │   │   │   └── activation_fn (SiLUActivation) 0
│   │   │   ├── input_layernorm (Phi3RMSNorm) 3.1K → weight:[3072]
│   │   │   ├── post_attention_layernorm (Phi3RMSNorm) 3.1K → weight:[3072]
│   │   │   ├── resid_attn_dropout (Dropout) 0
│   │   │   └── resid_mlp_dropout (Dropout) 0
│   │   ├── 2 (Phi3DecoderLayer) 50.34M
│   │   │   ├── self_attn (Phi3Attention) 12.58M
│   │   │   │   ├── o_proj (Linear4bit) 4.72M → weight:[4718592, 1]
│   │   │   │   └── qkv_proj (Linear4bit) 7.86M → weight:[7864320, 1]
│   │   │   ├── mlp (Phi3MLP) 37.75M
│   │   │   │   ├── gate_up_proj (Linear4bit) 25.17M → weight:[25165824, 1]
│   │   │   │   ├── down_proj (Linear4bit) 12.58M → weight:[12582912, 1]
│   │   │   │   └── activation_fn (SiLUActivation) 0
│   │   │   ├── input_layernorm (Phi3RMSNorm) 3.1K → weight:[3072]
│   │   │   ├── post_attention_layernorm (Phi3RMSNorm) 3.1K → weight:[3072]
│   │   │   ├── resid_attn_dropout (Dropout) 0
│   │   │   └── resid_mlp_dropout (Dropout) 0
│   │   ├── 3 (Phi3DecoderLayer) 50.34M
│   │   │   ├── self_attn (Phi3Attention) 12.58M
│   │   │   │   ├── o_proj (Linear4bit) 4.72M → weight:[4718592, 1]
│   │   │   │   └── qkv_proj (Linear4bit) 7.86M → weight:[7864320, 1]
│   │   │   ├── mlp (Phi3MLP) 37.75M
│   │   │   │   ├── gate_up_proj (Linear4bit) 25.17M → weight:[25165824, 1]
│   │   │   │   ├── down_proj (Linear4bit) 12.58M → weight:[12582912, 1]
│   │   │   │   └── activation_fn (SiLUActivation) 0
│   │   │   ├── input_layernorm (Phi3RMSNorm) 3.1K → weight:[3072]
│   │   │   ├── post_attention_layernorm (Phi3RMSNorm) 3.1K → weight:[3072]
│   │   │   ├── resid_attn_dropout (Dropout) 0
│   │   │   └── resid_mlp_dropout (Dropout) 0
│   │   ├── 4 (Phi3DecoderLayer) 50.34M
│   │   │   ├── self_attn (Phi3Attention) 12.58M
│   │   │   │   ├── o_proj (Linear4bit) 4.72M → weight:[4718592, 1]
│   │   │   │   └── qkv_proj (Linear4bit) 7.86M → weight:[7864320, 1]
│   │   │   ├── mlp (Phi3MLP) 37.75M
│   │   │   │   ├── gate_up_proj (Linear4bit) 25.17M → weight:[25165824, 1]
│   │   │   │   ├── down_proj (Linear4bit) 12.58M → weight:[12582912, 1]
│   │   │   │   └── activation_fn (SiLUActivation) 0
│   │   │   ├── input_layernorm (Phi3RMSNorm) 3.1K → weight:[3072]
│   │   │   ├── post_attention_layernorm (Phi3RMSNorm) 3.1K → weight:[3072]
│   │   │   ├── resid_attn_dropout (Dropout) 0
│   │   │   └── resid_mlp_dropout (Dropout) 0
│   │   ├── 5 (Phi3DecoderLayer) 50.34M
│   │   │   ├── self_attn (Phi3Attention) 12.58M
│   │   │   │   ├── o_proj (Linear4bit) 4.72M → weight:[4718592, 1]
│   │   │   │   └── qkv_proj (Linear4bit) 7.86M → weight:[7864320, 1]
│   │   │   ├── mlp (Phi3MLP) 37.75M
│   │   │   │   ├── gate_up_proj (

In [19]:
parser.compare_models(parsed_phi, parsed_tinyllama)

Metric                              Phi-4-mini-instruct   TinyLlama-1.1B-Chat-v1.0
--------------------------------------------------------------------------------
Total params                                   2225.41M                    615.61M
Trainable params                                614.80M                    131.16M
Dtype(s)                      torch.bfloat16, torch.uint8  torch.bfloat16, torch.uint8
Device(s)                                        cuda:0                     cuda:0

Layer-type counts:
  Dropout                                            64                          0
  Embedding                                           1                          1
  Linear                                              1                          1
  Linear4bit                                        128                        154
  LlamaAttention                                      0                         22
  LlamaDecoderLayer                                   0          

# Task 2


In [20]:
!pip install pydantic

In [7]:
from pydantic_settings import BaseSettings

class TrainingConfig(BaseSettings):
    # Model
    model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    # Dataset
    dataset_name: str = "cnn_dailymail"
    dataset_version: str = "3.0.0"
    train_split: str = "train[:200]"
    test_size: float = 0.1

    # Tokenization
    max_length: int = 256

    # LoRA
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05

    # Training
    batch_size: int = 1
    epochs: int = 1
    learning_rate: float = 2e-5

    # Output
    output_dir: str = "./lora-results"
    adapter_path: str = "./lora_adapter"

In [8]:
from datasets import load_dataset

class DataHandler:
    def __init__(self, config: TrainingConfig):
        self.config = config
        self.train_dataset = None
        self.eval_dataset = None
        self.eval_dataset_for_generation = None

    def load(self):
        dataset = load_dataset(
            self.config.dataset_name,
            self.config.dataset_version,
            split=self.config.train_split
        )

        def format_example(example):
            return {
                "text": f"""### Instruction:
Summarize the following article

### Input:
{example['article'][:800]}

### Response:
{example['highlights']}"""
            }

        dataset = dataset.map(format_example)
        dataset = dataset.train_test_split(test_size=self.config.test_size)

        self.train_dataset = dataset["train"]
        self.eval_dataset = dataset["test"]
        # Make an independent copy for generation BEFORE tokenization
        # This ensures it retains the 'text' column for evaluation/generation
        self.eval_dataset_for_generation = dataset["test"].select(range(len(dataset["test"]))) # Creates a new Dataset object

    def tokenize(self, tokenizer):
        def tokenize_fn(example):
            return tokenizer(
                example["text"],
                truncation=True,
                padding="max_length",
                max_length=self.config.max_length
            )

        self.train_dataset = self.train_dataset.map(tokenize_fn, batched=True)
        self.eval_dataset = self.eval_dataset.map(tokenize_fn, batched=True)

        self.train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])
        self.eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

In [9]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from rich.table import Table
from rich import print
import random

class LoRATrainer:

    def __init__(self, config, model_handler, data_handler):
        self.config = config
        self.model_handler = model_handler
        self.data_handler = data_handler

        self.model = model_handler.get_model()
        self.tokenizer = model_handler.tokenizer
        self.base_model = None

    # -------------------------
    def apply_lora(self):
        lora_config = LoraConfig(
            r=self.config.lora_r,
            lora_alpha=self.config.lora_alpha,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=self.config.lora_dropout,
            bias="none",
            task_type="CAUSAL_LM"
        )

        self.model = get_peft_model(self.model, lora_config)

        # update handler reference
        self.model_handler.model = self.model

        self.model.print_trainable_parameters()

    # -------------------------
    def train(self):
        training_args = TrainingArguments(
            output_dir=self.config.output_dir,
            per_device_train_batch_size=self.config.batch_size,
            per_device_eval_batch_size=self.config.batch_size,
            num_train_epochs=self.config.epochs,
            learning_rate=self.config.learning_rate,
            logging_steps=5,
            save_steps=50,
            eval_strategy="steps",
            eval_steps=50,
            fp16=True,
            report_to="none"
        )

        data_collator = DataCollatorForLanguageModeling(
            tokenizer=self.tokenizer,
            mlm=False
        )

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=self.data_handler.train_dataset,
            eval_dataset=self.data_handler.eval_dataset,
            data_collator=data_collator
        )

        trainer.train()

    # -------------------------
    def save(self):
        self.model.save_pretrained(self.config.adapter_path)
        self.tokenizer.save_pretrained(self.config.adapter_path)

    # -------------------------
    def generate(self, prompt):
        return self.generate_with_model(self.model, prompt)

    def _extract_parts(self, text):
        parts = text.split("### Response:")
        input_part = parts[0]
        gt = parts[1].strip() if len(parts) > 1 else ""
        return input_part, gt


    def _short(self, text, width=120):
        from textwrap import shorten
        return shorten(text.replace("\n", " "), width=width, placeholder="...")

    def load_base_model(self):
        from transformers import AutoModelForCausalLM

        self.base_model = AutoModelForCausalLM.from_pretrained(
            self.config.model_name,
            device_map="auto"
        )

    def generate_with_model(self, model, prompt):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            pad_token_id=self.tokenizer.eos_token_id
        )

        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

        text = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)

        return text.strip()

    def _extract_parts(self, text):
        response_marker = "### Response:"

        if response_marker in text:
            prompt_input_part = text.split(response_marker)[0].strip()
            prompt = prompt_input_part + "\n\n" + response_marker + "\n"
            gt = text.split(response_marker)[1].strip()
        else:
            prompt = text
            gt = ""

        return prompt, gt

    def evaluate_samples(self, num_samples=3, use_rich=True):
        if self.base_model is None:
            print(" Base model not found. Loading automatically...")
            self.load_base_model()

        dataset = self.data_handler.eval_dataset_for_generation

        #  Random sampling
        indices = random.sample(range(len(dataset)), num_samples)

        from rich.table import Table
        from rich import print as rprint

        for i, idx in enumerate(indices):
            sample = dataset[idx]

            prompt, ground_truth = self._extract_parts(sample["text"])

            base_output = self.generate_with_model(self.base_model, prompt)
            lora_output = self.generate(prompt)

            if use_rich:
                table = Table(title=f"Random Sample {i} (idx={idx})", show_lines=True)

                table.add_column("Type", style="cyan", width=15)
                table.add_column("Output", style="white", overflow="fold")

                table.add_row("Ground Truth", ground_truth)
                table.add_row("Base Model", base_output)
                table.add_row("LoRA Model", lora_output)

                rprint(table)
            else:
                print(f"\n===== SAMPLE {i} (idx={idx}) =====")
                print("GT:", ground_truth)
                print("BASE:", base_output)
                print("LORA:", lora_output)



In [10]:
config = TrainingConfig()

# Data
data_handler = DataHandler(config)
data_handler.load()

# Model
model_handler = HuggingFaceModelHandler(config.model_name)
model_handler.load_model()

# Tokenize (uses tokenizer from model handler)
data_handler.tokenize(model_handler.tokenizer)

# Trainer
trainer = LoRATrainer(config, model_handler, data_handler)
trainer.load_base_model()






Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <class '__main__.TrainingConfig'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.12/dist-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <class '__main__.TrainingConfig'>: __main__.TrainingConfig has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
Parameter 'function'=<function DataHandler.tokenize.<locals>.tokenize_fn at 0x7bc484c4e8e0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be sh

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [16]:
trainer.apply_lora()
trainer.train()
trainer.save()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


Step,Training Loss,Validation Loss
50,2.170944,2.214324
100,2.166713,2.150230
150,2.165431,2.116079


In [17]:
trainer.evaluate_samples(num_samples=3)

                                              Random Sample 0 (idx=3)                                              
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Type            ┃ Output                                                                                        ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Ground Truth    │ Man posted photos on the Internet of himself sexually abusing underage boys .                 │
│                 │ Computer experts managed to undo digital masking to reveal the man .                          │
│                 │ Man abused 12 boys in Vietnam and Cambodia .                                                  │
├─────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────┤
│ Base Model      │ The Interpol has taken a unique step by making a global appeal for help to identify a man     │
│                 │ from digitally reconstructed photos taken from the Internet. The man's face was disguised by  │
│                 │ digital alteration, but the images were capable of being restored, according to a bulletin    │
│                 │ from Interpol. The man's identity and nationality are still unknown. The Interpol has been    │
│                 │ unable to determine the man's identity or nationality. The man's face was disgu               │
├─────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────┤
│ LoRA Model      │ The man's face was disguised by digital alteration, but the images were capable of being      │
│                 │ restored, according to a bulletin from Interpol -- the international police agency based in   │
│                 │ Lyon, France. Interpol Secretary General Ronald K. Noble said the pictures have been on the   │
│                 │ the Internet for several years, but investigators have been unable to determine the man's     │
│                 │ identity or nationality. "We have tried all other means to identify and to bring him to       │
│                 │ justice, but                                                                                  │
└─────────────────┴───────────────────────────────────────────────────────────────────────────────────────────────┘

                                              Random Sample 1 (idx=0)                                              
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Type            ┃ Output                                                                                        ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Ground Truth    │ Official: Streets bustling, workers returning, markets "back like gangbusters"                │
│                 │ Troop surge, citizens groups, Mehdi Army cease-fire all help security situation .             │
│                 │ Iranian weapons, fighters still posing problems in northeastern Baghdad .                     │
│                 │ Commander says more families will return when basic services fully restored .                 │
├─────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────┤
│ Base Model      │ The article does not provide any specific details about the threats posed by Iranian-backed   │
│                 │ militias in the area.                                                                         │
├─────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────┤
│ LoRA Model      │ "The situation is still dangerous, but it's dramatically improving," Farris said. "We're      │
│                 │ seeing a lot of progress." The military official said the improvements are due to the efforts │
│                 │ of Iraqi civilians, who have helped the troops improve security in their neighborhoods. "The  │
│                 │ Iraqi people are doing a great job," Farris said. "They're helping us. They're helping us     │
│                 │ with the security. They're helping us with the                                                │
└─────────────────┴───────────────────────────────────────────────────────────────────────────────────────────────┘

                                              Random Sample 2 (idx=8)                                              
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Type            ┃ Output                                                                                        ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Ground Truth    │ NEW: NFL chief, Atlanta Falcons owner critical of Michael Vick's conduct .                    │
│                 │ NFL suspends Falcons quarterback indefinitely without pay .                                   │
│                 │ Vick admits funding dogfighting operation but says he did not gamble .                        │
│                 │ Vick due in federal court Monday; future in NFL remains uncertain .                           │
├─────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────┤
│ Base Model      │ The National Football League has indefinitely suspended Atlanta Falcons quarterback Michael   │
│                 │ Vick without pay, officials with the league said Friday. The NFL star is set to appear in     │
│                 │ court on Monday. A judge will have the final say on a plea deal. Earlier, Vick admitted to    │
│                 │ participating in a dogfighting ring as part of a plea agreement with federal prosecutors in   │
│                 │ Virginia. "Your admitted conduct was not only illegal, but also cruel and reprehensible       │
├─────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────┤
│ LoRA Model      │ The National Football League has indefinitely suspended Atlanta Falcons quarterback Michael   │
│                 │ Vick without pay, officials with the league said Friday. The league said in a statement that  │
│                 │ Vick's suspension was "in accordance with the NFL's policy on player conduct." Earlier, Vick  │
│                 │ admitted to participating in a dogfighting ring as part of a plea agreement with federal      │
│                 │ prosecutors in Virginia. "Your admitted conduct was not only illegal, but also cruel and      │
└─────────────────┴───────────────────────────────────────────────────────────────────────────────────────────────┘

# Task 3

In [18]:
!pip install rouge_score

In [16]:
import torch, copy, gc, time, random
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from rouge_score import rouge_scorer
import pandas as pd


class ModelEvaluator:

    def __init__(self, config, model_handler, data_handler):
        self.config = config
        self.model_handler = model_handler
        self.data_handler = data_handler

        self.tokenizer = model_handler.tokenizer
        self.BASE_MODEL = config.model_name
        self.ADAPTER_PATH = config.adapter_path
        self.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

        self.bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        self.scorer = rouge_scorer.RougeScorer(
            ["rouge1", "rouge2", "rougeL"], use_stemmer=True
        )

    def average_weights(self, sd_a, sd_b, alpha=0.5):
        merged = copy.deepcopy(sd_a)

        for k in merged:
            if k in sd_b and sd_a[k].shape == sd_b[k].shape \
               and sd_a[k].dtype.is_floating_point:
                merged[k] = (1 - alpha) * sd_a[k] + alpha * sd_b[k]

        return merged


    def create_weight_averaged_model(self):
        print("Creating weight averaged model...")

        base = AutoModelForCausalLM.from_pretrained(
            self.BASE_MODEL,
            dtype=torch.float16,
            device_map="cpu"
        )

        lora_merged = AutoModelForCausalLM.from_pretrained(
            "./tinyllama-lora-merged",
            dtype=torch.float16,
            device_map="cpu"
        )


        for (name, param_base), (_, param_lora) in zip(
            base.named_parameters(),
            lora_merged.named_parameters()
        ):
            if param_base.shape == param_lora.shape and param_base.dtype.is_floating_point:
                param_base.data = 0.5 * param_base.data + 0.5 * param_lora.data

        # store result
        self.model_avg = base

        print("Weight averaged model ready")

        del lora_merged
        gc.collect()


    def create_lora_merged_model(self):
        print("Creating LoRA merged model...")

        base = AutoModelForCausalLM.from_pretrained(
            self.BASE_MODEL,
            dtype=torch.float16,
            device_map="cpu"
        )

        peft_model = PeftModel.from_pretrained(base, self.ADAPTER_PATH)

        # Merge LoRA → base
        merged_model = peft_model.merge_and_unload()


        self.model_lora_merged = merged_model

        print("LoRA merged model ready")

        del peft_model
        gc.collect()


    def load_model(self, name):

        if name == "Base":
            # Load a fresh base model, don't use the potentially modified self.model_handler.get_model()
            return AutoModelForCausalLM.from_pretrained(
                self.BASE_MODEL,
                quantization_config=self.bnb_cfg, # Use the same bnb config for consistency
                device_map="auto"
            ).eval()

        elif name == "Base + Adapter":
            base = AutoModelForCausalLM.from_pretrained(
                self.BASE_MODEL,
                quantization_config=self.bnb_cfg,
                device_map="auto"
            )
            return PeftModel.from_pretrained(base, self.ADAPTER_PATH).eval()

        elif name == "Weight Averaged":
            return self.model_avg.to(self.DEVICE)

        elif name == "LoRA Merged":
            if not hasattr(self, "model_lora_merged"):
                raise ValueError("LoRA merged model not created yet!")
            return self.model_lora_merged.to(self.DEVICE)


    def get_perplexity(self, model, texts):
        losses = []

        for text in texts:
            enc = self.tokenizer(
                text, return_tensors="pt",
                truncation=True, max_length=128
            ).to(self.DEVICE)

            with torch.no_grad():
                loss = model(**enc, labels=enc["input_ids"]).loss

            losses.append(loss.item())

        return round(torch.exp(torch.tensor(sum(losses)/len(losses))).item(), 2)


    def get_rouge(self, model, pairs, n=20):
        r1, r2, rl = [], [], []

        for prompt_dict, ref in pairs[:n]:
            enc = self.tokenizer(
                prompt_dict["text"],
                return_tensors="pt",
                truncation=True,
                max_length=512
            ).to(self.DEVICE)

            with torch.no_grad():
                out = model.generate(
                    **enc,
                    max_new_tokens=80,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            hyp = self.tokenizer.decode(out[0], skip_special_tokens=True)

            s = self.scorer.score(ref, hyp)

            r1.append(s["rouge1"].fmeasure)
            r2.append(s["rouge2"].fmeasure)
            rl.append(s["rougeL"].fmeasure)

        avg = lambda x: round(sum(x)/len(x), 4)
        return avg(r1), avg(r2), avg(rl)


    def measure_latency(self, model, pairs, n=10):
        times = []

        for prompt_dict, _ in pairs[:n]:
            enc = self.tokenizer(prompt_dict["text"], return_tensors="pt").to(self.DEVICE)

            start = time.time()

            with torch.no_grad():
                _ = model.generate(**enc, max_new_tokens=80)

            end = time.time()
            times.append(end - start)

        return round((sum(times)/len(times))*1000, 2)


    def evaluate(self):

        dataset = self.data_handler.eval_dataset_for_generation

        eval_pairs = []

        for x in dataset[:30]:
            text = x["text"] if isinstance(x, dict) else x

            eval_pairs.append(
                ({"text": text}, text.split("### Response:")[-1].strip())
            )

        ref_summaries = [pair[1] for pair in eval_pairs[:5]]

        models = ["Base", "Base + Adapter", "LoRA Merged"]

        rows = []
        samples_output = {}

        for name in models:
            print(f"\n🔹 Processing {name}...")

            # ======================
            # 1. LOAD MODEL
            # ======================
            model = self.load_model(name)

            # ======================
            # 2. METRICS
            # ======================
            ppl = self.get_perplexity(model, ref_summaries)
            r1, r2, rl = self.get_rouge(model, eval_pairs, n=10)
            latency = self.measure_latency(model, eval_pairs, n=5)

            rows.append({
                "Model": name,
                "Perplexity ↓": ppl,
                "ROUGE-1 ↑": r1,
                "ROUGE-2 ↑": r2,
                "ROUGE-L ↑": rl,
                "Latency (ms) ↓": latency
            })

            # ======================
            # 3. SAMPLE GENERATION
            # ======================
            sample_indices = random.sample(range(len(dataset)), 2)
            samples_output[name] = []

            for idx in sample_indices:
                sample = dataset[idx]

                text = sample["text"] if isinstance(sample, dict) else sample

                prompt = text.split("### Response:")[0].strip() + "\n\n### Response:\n"

                inputs = self.tokenizer(prompt, return_tensors="pt").to(model.device)
                out = model.generate(**inputs, max_new_tokens=50)

                text = self.tokenizer.decode(out[0], skip_special_tokens=True)

                samples_output[name].append({
                    "idx": idx,
                    "output": text
                })

            # ======================
            # 4. UNLOAD MODEL (CRITICAL)
            # ======================
            del model
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(2)

        # ======================
        # 5. PRINT RESULTS
        # ======================
        df = pd.DataFrame(rows).set_index("Model")
        print("\n===== METRICS =====\n")
        print(df.to_string())

        print("\n===== SAMPLE OUTPUTS =====\n")
        for model_name, outputs in samples_output.items():
            print(f"\n🔸 {model_name}")
            for item in outputs:
                print(f"\nSample {item['idx']}:\n{item['output']}\n")

        return df


    def generate_samples(self, num_samples=2):

        dataset = self.data_handler.eval_dataset_for_generation
        indices = random.sample(range(len(dataset)), num_samples)

        models = [
            "Base",
            "Base + Adapter",
            "LoRA Merged"
        ]

        for idx in indices:
            sample = dataset[idx]
            text = sample["text"] if isinstance(sample, dict) else sample

            prompt = text.split("### Response:")[0].strip() + "\n\n### Response:\n"

            print(f"\n===== SAMPLE (idx={idx}) =====\n")

            for name in models:
                model = self.load_model(name)

                inputs = self.tokenizer(prompt, return_tensors="pt").to(model.device)

                out = model.generate(**inputs, max_new_tokens=80)

                text = self.tokenizer.decode(out[0], skip_special_tokens=True)

                print(f"\n{name}:\n{text}\n")

                del model
                torch.cuda.empty_cache()
                gc.collect()

In [17]:
evaluator = ModelEvaluator(config, model_handler, data_handler)

evaluator.create_lora_merged_model()
# evaluator.create_weight_averaged_model()

df = evaluator.evaluate()

evaluator.generate_samples(num_samples=2)

Creating LoRA merged model...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LoRA merged model ready

🔹 Processing Base...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Base:
### Instruction:
Summarize the following article

### Input:
JENA, Louisiana (CNN)  -- Charges against Bryant Purvis, one of the six black students accused of being involved in
beating a white student, were reduced to second degree aggravated battery during his arraignment Wednesday morning.
Bryant Purvis says he is focusing on his studies and practicing basketball. Purvis, who was facing charges of 
second-degree attempted murder and conspiracy, entered a not guilty plea to the reduced charges in the LaSalle 
Parish Courthouse in Jena. Charges have now been reduced against at least five of the students in the racially 
charged "Jena 6" case. Charges against Jesse Ray Beard, who was 14 at the time of the alleged crime, are 
unavailable because he's a juvenile. Civil rights leaders Martin Luther King III and Al Sharpton led more than 
15,000 marchers to J

### Response:
The article states that charges against Jesse Ray Beard, who was 14 at the time of the alleged crime, are 
unavailable because he's a juvenile.

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Base + Adapter:
### Instruction:
Summarize the following article

### Input:
JENA, Louisiana (CNN)  -- Charges against Bryant Purvis, one of the six black students accused of being involved in
beating a white student, were reduced to second degree aggravated battery during his arraignment Wednesday morning.
Bryant Purvis says he is focusing on his studies and practicing basketball. Purvis, who was facing charges of 
second-degree attempted murder and conspiracy, entered a not guilty plea to the reduced charges in the LaSalle 
Parish Courthouse in Jena. Charges have now been reduced against at least five of the students in the racially 
charged "Jena 6" case. Charges against Jesse Ray Beard, who was 14 at the time of the alleged crime, are 
unavailable because he's a juvenile. Civil rights leaders Martin Luther King III and Al Sharpton led more than 
15,000 marchers to J

### Response:
The article states that charges against Jesse Ray Beard, who was 14 at the time of the alleged crime, are 
unavailable because he's a juvenile.

LoRA Merged:
### Instruction:
Summarize the following article

### Input:
JENA, Louisiana (CNN)  -- Charges against Bryant Purvis, one of the six black students accused of being involved in
beating a white student, were reduced to second degree aggravated battery during his arraignment Wednesday morning.
Bryant Purvis says he is focusing on his studies and practicing basketball. Purvis, who was facing charges of 
second-degree attempted murder and conspiracy, entered a not guilty plea to the reduced charges in the LaSalle 
Parish Courthouse in Jena. Charges have now been reduced against at least five of the students in the racially 
charged "Jena 6" case. Charges against Jesse Ray Beard, who was 14 at the time of the alleged crime, are 
unavailable because he's a juvenile. Civil rights leaders Martin Luther King III and Al Sharpton led more than 
15,000 marchers to J

### Response:
The article does not provide any information on the charges against Jesse Ray Beard, who was 14 at the time of the 
alleged crime.